## Extract Features for training the model in notebook 05

### Modeling Notes
I'm only using phase 2 because it represents the task phase
I will have to restrict the gamma band to 30-40 Hz because phase 2 already has a band pass filter applied. 

### Import libaries

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import mne

PROCESSED_DIR = Path("../data/processed")

BANDS = {
     "theta": (4, 8),
     "alpha": (8, 12),
     "beta_low": (12, 18),
     "beta_high": (18, 25),
     "gamma": (25, 40) 
}



### Make an extraction feature function
It will apply to all the bands

In [ ]:
def extract_band_features(epochs_path, bands=BANDS):
     """ Takes in the file path and desired dictionary of 
     power bands to output a DataFrame of the features of a 
     set of epochs. 
     Parameters:
          epochs_path = string of file directory to processed epochs
          bands = dictionary of power bands to be check it out
          
     Returns:
          DataFrame of features

     """
     # replicating code fro 02_preprocessing.ipynb
     epochs = mne.read_epochs(epochs_path, preload=True)
     ch_names = epochs.ch_names
     
     spectrum = epochs.compute_psd(method="welch", fmin=4, fmax=40, verbose=False)
     psds, freqs = spectrum.get_data(return_freqs=True)  # shape: (n_epochs, n_channels, n_freqs)

     n_epochs = psds.shape[0]
     rows = [dict() for e in range(n_epochs)]

     for band_name, (fmin, fmax) in bands.items():
          band_mask = (freqs >= fmin) & (freqs <= fmax)
          band_power = psds[:, :, band_mask].mean(axis=2)  # shape: (n_epochs, n_channels)
          
          # convert band power to dbs in order make variance more clear
          band_power_db = 10 * np.log10(band_power * 1e12 + 1e-8)
          for ch_idx, ch_name in enumerate(ch_names):
               col = f"{ch_name}_{band_name}"
               for epoch_idx in range(n_epochs):
                    rows[epoch_idx][col] = band_power_db[epoch_idx, ch_idx]

     eps = 1e-8
     # Making a frontal theta/beta ratio
     # Channel EEG.AF3 and EEG.AF4 are frontal channels
     # Helps model be more accurate by acting as a stress marker
     frontal = [c for c in ("EEG.AF3", "EEG.AF4") if c in ch_names]
     if len(frontal) == 2:
          theta_mask = (freqs >= bands["theta"][0]) & (freqs <= bands["theta"][1])
          beta_mask = (freqs >= bands["beta_low"][0]) & (freqs <= bands["beta_high"][1])
          
          frontal_idx = [ch_names.index(c) for c in frontal]
          frontal_theta = psds[:, frontal_idx][:, :, theta_mask].mean(axis=(1,2))
          frontal_beta = psds[:, frontal_idx][:, :, beta_mask].mean(axis=(1,2))
          frontal_ratio = frontal_theta / (frontal_beta + eps)
          frontal_ratio = np.nan_to_num(frontal_ratio, nan=0.0, posinf=100.0, neginf=0.0)
          frontal_ratio = np.clip(frontal_ratio, 0, 100)
          
          for epoch_idx in range(n_epochs):
               rows[epoch_idx]["frontal_theta_beta_ratio"] = frontal_ratio[epoch_idx]
             
     return pd.DataFrame(rows)
        

### Run function through every segment
#### Keeping only the task-phase epochs

In [ ]:
all_rows = []
epoch_files = sorted(PROCESSED_DIR.glob("*-epo.fif"))
print(f"Found {len(epoch_files)} processed segment files from processed data folder")

skipped_non_task = 0
# replicating code from 03_matlab_integration notebook
for fif_path in epoch_files:
     stem = fif_path.stem.replace("-epo", "")
     parts = stem.split("_")
     subject = f"{parts[0]}_{parts[1]}"
     test = int(parts[2].replace("test", ""))
     phase = int(parts[3].replace("phase", ""))
     
     if phase != 2:
          skipped_non_task += 1
          continue # only the task-phase (2) epochs get parsed 
     
     features = extract_band_features(fif_path)
     features.insert(0, "subject", subject)
     features.insert(1, "test", test)
     features.insert(2, "phase", phase)
     all_rows.append(features)

# combining the rows into one large DataFrame
full_features = pd.concat(all_rows, ignore_index=True)
print(f"Skipped {skipped_non_task} non-task-phase files")
print(f"Finalized feature table: {full_features.shape[0]} and {full_features.shape[1]} columns")


### Double checking that the features are balanced out

In [ ]:
print("Class balancing (test is workload level):\n")
print(full_features["test"].value_counts().sort_index())

print(f"There are {full_features["subject"].nunique()} subjects represented\n")

n_nans = full_features.isna().sum().sum() # finding the number of NaN values
print(f"\nTotal NaN values: {n_nans}")
if n_nans > 0:
     print("Coulmns with NaNs:")
     nan_cols = full_features.column[full_features.isna().any()].tolist()
     print(nan_cols)
     
full_features.to_csv(PROCESSED_DIR / "full_features.csv", index=False)
print(f"Saved full_features.csv to {PROCESSED_DIR / 'full_features.csv'}")


In [ ]:
full_features.head()

### Findings
143 epochs

Skipped 95 non-task-phase files

Finalized feature table: 20769 and 74 columns

0 NaN values

test
<ul>
<li>1    7157</li>
<li>2    6694</li>
<li>3    6918</li>
</ul>
REMINDER: Frontal theta/beta got included in the full_features.csv